In [13]:
# Basic RAG Retrieval - What is machine learning?
# Semantic Understanding (Not exact match) - Explain deep learning in simple terms
# Another Knowledge Query - What does NLP do?
# Multi-step Output Check - Tell me about machine learning and create questions
# Memory Test (Very Important) - step 1 : My name is Rahul  -  step 2: what is my name
# Cross-question Memory - step 1: What is deep learning? - step 2: what did i ask before this?
# Tool Usage Test (Important) - What is 25 * 4?
# Out-of-scope Query - what is block chain?
# Combined Query - Explain NLP and give me quiz questions
# Complex Natural Query - How is machine learning used in real life?

In [1]:
!pip install langchain langchain-community langchain-google-genai faiss-cpu sentence-transformers

In [2]:
import os

# ----------------------------
# Imports
# ----------------------------

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [6]:
# ----------------------------
# API Key
# ----------------------------

#os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [8]:
# ----------------------------
# Embeddings + Vector DB
# ----------------------------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

documents = [
    Document(page_content="Machine learning is used in recommendation systems and fraud detection."),
    Document(page_content="Deep learning uses neural networks and is useful in image recognition."),
    Document(page_content="NLP enables machines to understand human language."),
]

vector_db = FAISS.from_documents(documents, embedding_model)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})


# ----------------------------
# Tool
# ----------------------------

@tool
def calculator(expression: str) -> str:
    """Evaluate math expressions."""
    return str(eval(expression))


tools = [calculator]


# ----------------------------
# LLM
# ----------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

llm_with_tools = llm.bind_tools(tools)


# ----------------------------
# Helper
# ----------------------------

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# ----------------------------
# Step 1: Retrieval
# ----------------------------

retrieval_chain = retriever | RunnableLambda(format_docs)


# ----------------------------
# Step 2: Summarization
# ----------------------------

summary_prompt = ChatPromptTemplate.from_template(
"""
Summarize the following content in 3 sentences:

{context}
"""
)

summary_chain = summary_prompt | llm


# ----------------------------
# Step 3: Quiz Generation
# ----------------------------

quiz_prompt = ChatPromptTemplate.from_template(
"""
Based on the summary below, generate 2 quiz questions:

{summary}
"""
)

quiz_chain = quiz_prompt | llm


# ----------------------------
# Step 4: Combine Steps
# ----------------------------

def multi_step_pipeline(input_data):

    # ✅ Extract correct string
    if isinstance(input_data, dict):
        question = input_data["question"]
    else:
        question = input_data

    # STEP 1: Retrieve (FIXED)
    docs = retriever.invoke(question)
    context = format_docs(docs)

    # STEP 2: Summarize
    summary = summary_chain.invoke({"context": context}).content

    # STEP 3: Quiz
    quiz = quiz_chain.invoke({"summary": summary}).content

    return f"Summary:\n{summary}\n\nQuiz:\n{quiz}"


multi_step_chain = RunnableLambda(multi_step_pipeline)


# ----------------------------
# Final Prompt
# ----------------------------

final_prompt = ChatPromptTemplate.from_template(
"""
You are a helpful AI assistant.

Chat History:
{chat_history}

User Question:
{question}

Here is processed information:
{result}

Provide a helpful final answer.
"""
)


# ----------------------------
# Final Chain
# ----------------------------

final_chain = (
    {
        "result": multi_step_chain,
        "question": RunnablePassthrough(),
        "chat_history": RunnablePassthrough()
    }
    | final_prompt
    | llm_with_tools
)


# ----------------------------
# Memory
# ----------------------------

store = {}

def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


agent = RunnableWithMessageHistory(
    final_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# ----------------------------
# Chat Loop
# ----------------------------

while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    response = agent.invoke(
        {"question": query},
        config={"configurable": {"session_id": "multi_agent"}}
    )

    print("\nAgent:\n", response.content)


You: Tell me about machine learning and create questions

Agent:
 [{'type': 'text', 'text': "Here's a summary of machine learning and some quiz questions based on it:\n\n**Summary:**\nMachine learning is a field of artificial intelligence that focuses on enabling systems to learn from data without explicit programming. It is utilized in various applications, including recommendation systems and fraud detection. Natural Language Processing (NLP) is a technology that enables machines to understand human language. Both machine learning and NLP are fundamental for developing intelligent systems.\n\n**Quiz Questions:**\n\n1.  Which of the following is an application of Machine Learning mentioned in the summary?\n    a) Building physical robots\n    b) Recommendation systems\n    c) Developing new programming languages\n    d) Designing computer hardware\n\n2.  What is the primary focus of Natural Language Processing (NLP)?\n    a) Creating visual graphics\n    b) Analyzing large datasets f

In [12]:
while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    # Step 1: First LLM call
    response = agent.invoke(
        {"question": query},
        config={"configurable": {"session_id": "user1"}}
    )

    # Step 2: Check if tool is called
    if hasattr(response, "tool_calls") and response.tool_calls:

        tool_call = response.tool_calls[0]

        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        # Step 3: Execute tool
        if tool_name == "calculator":
            tool_result = calculator.invoke(tool_args["expression"])

        # Step 4: Send tool result back to LLM
        final_prompt = f"""
User question: {query}

Tool result: {tool_result}

Give the final answer.
"""

        final_response = llm.invoke(final_prompt)

        print("\nAgent:", final_response.content)

    else:
        # No tool used
        print("\nAgent:", response.content)


You: 99 * 3 = ?

Agent: 297

You: what is machine learning and give me quiz questions

Agent: [{'type': 'text', 'text': 'Machine learning is a field of artificial intelligence that allows systems to learn from data, identify patterns, and make decisions with minimal human intervention. It is utilized in various applications such as recommendation systems and fraud detection. Natural Language Processing (NLP) is a key technology within AI that enables machines to understand, interpret, and generate human language. Both machine learning and NLP are fundamental for developing intelligent systems.\n\nHere are two quiz questions based on the summary:\n\n1.  According to the summary, name two applications where machine learning is utilized.\n2.  Which technology enables machines to understand human language?\n    a) Machine Learning\n    b) Fraud Detection\n    c) Natural Language Processing (NLP)\n    d) Recommendation Systems', 'extras': {'signature': 'CvcBAb4+9vua8sM5a30RA85twJaufjhqvBMJ